# PLINDER linked-apo pockets: atoms + experimental density

Browse the validated ligand-free apo pockets produced by `voxbind/dataset/plinder/06_build_apo_pairs.py`. The 3D view overlays:

- apo protein heavy-atom coordinates, coloured by element;
- the experimental apo 2Fo–Fc density map in the deposited apo coordinate frame;
- the transformed holo ligand as a **reference anchor only** (red crosses). It is not present in the apo structure or model input.

The coordinate table also reports the experimental density z-score sampled at every pocket atom.

In [1]:
from pathlib import Path

import gemmi
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scipy.ndimage
import torch
from IPython.display import display


def find_voxbind_root(start=Path.cwd().resolve()):
    """Find the repository root whether Jupyter starts here or above it."""
    for path in (start, *start.parents):
        if (path / "voxbind" / "dataset").is_dir() and (path / "notebook").is_dir():
            return path
    raise RuntimeError(f"Could not locate the VoxBind root from {start}")


ROOT = find_voxbind_root()
PRETRAIN = ROOT / "voxbind" / "dataset" / "data" / "pretrain"
PAIR_FILE = PRETRAIN / "plinder_v2_apo_pairs.pt"
PAIR_INDEX_FILE = PRETRAIN / "plinder_v2_apo" / "pair_index.csv"

if not PAIR_FILE.exists():
    raise FileNotFoundError(
        f"Missing {PAIR_FILE}. Run:\n"
        "  cd voxbind && python dataset/plinder/06_build_apo_pairs.py --version v2"
    )

pairs = torch.load(PAIR_FILE, map_location="cpu", weights_only=False)
pair_index = pd.read_csv(PAIR_INDEX_FILE)
print(f"Loaded {len(pairs):,} validated apo/holo pairs")
print(f"Unique experimental apo PDBs: {pair_index.apo_pdb_id.nunique():,}")
display(pair_index.head())

Loaded 1,329 validated apo/holo pairs
Unique experimental apo PDBs: 179


,alignment_coverage,alignment_identity,apo_chain,apo_id,apo_pdb_id,apo_resolution,binding_site_rmsd,holo_chain,n_alignment_atoms,n_apo_pocket_atoms,nearby_hetero_atoms,nearby_organic_atoms,pair_id,pocket_density_coverage,pocket_density_mean_z,pocket_fident,pocket_lddt,reason,reference_system_id,status
0,1.0,1.0,B,1b7y_B,1b7y,2.5,0.344249,B,396,338,0,0,3hfz__1__1.B__1.D__apo_1b7y_B,0.973373,2.851266,100.0,100.0,ok,3hfz__1__1.B__1.D,ok
1,1.0,1.0,B,1b7y_B,1b7y,2.5,0.344249,B,396,338,0,0,3hfz__1__2.B__2.D__apo_1b7y_B,0.973373,2.851266,100.0,100.0,ok,3hfz__1__2.B__2.D,ok
2,1.0,1.0,B,1d09_B,1d09,2.1,1.037715,B,196,283,0,0,4fyv__1__1.B__1.G__apo_1d09_B,0.830389,1.232072,100.0,88.0,ok,4fyv__1__1.B__1.G,ok
3,1.0,1.0,B,1d09_B,1d09,2.1,1.175960,D,208,288,0,0,4fyv__1__1.D__1.I__apo_1d09_B,0.843750,1.260750,100.0,89.0,ok,4fyv__1__1.D__1.I,ok
4,1.0,1.0,B,1d09_B,1d09,2.1,1.037715,B,196,283,0,0,4fyv__1__2.B__2.G__apo_1d09_B,0.830389,1.232072,100.0,88.0,ok,4fyv__1__2.B__2.G,ok


## Select a pocket

`PAIR_NUMBER` follows the saved pair order. A 16 Å density box (`40 × 0.40 Å`) is responsive in Plotly; increase `GRID_SIZE` or reduce `VOXEL_A` for a finer rendering.

In [2]:
PAIR_NUMBER = 0       # 0 <= PAIR_NUMBER < len(pairs)
GRID_SIZE = 40        # sampled points per axis
VOXEL_A = 0.40        # Å per sampled point
DENSITY_SIGMA = 1.0   # positive-density isosurface threshold
SHOW_ANCHOR = True    # transferred holo ligand; reference only, not apo input

In [3]:
POCKET_ELEMENTS = np.array(["C", "O", "N", "S"])
ELEMENT_COLORS = {"C": "#8c8c8c", "O": "#e74c3c", "N": "#3b82f6", "S": "#f2c94c"}
ELEMENT_SIZES = {"C": 4.0, "O": 4.4, "N": 4.4, "S": 5.2}


def as_numpy(value):
    return value.detach().cpu().numpy() if torch.is_tensor(value) else np.asarray(value)


def load_ccp4_grid(path):
    """Load a CCP4 map and its Cartesian-to-fractional row-vector transform."""
    density_map = gemmi.read_ccp4_map(str(path))
    density_map.setup(float("nan"))
    grid = density_map.grid
    values = np.array(grid, dtype=np.float32)
    if not np.isfinite(values).all() or values.std() < 1e-8:
        raise ValueError(f"Invalid/flat density map: {path}")
    orthogonal = np.asarray(grid.unit_cell.orth.mat.tolist(), dtype=np.float64)
    cartesian_to_fractional = np.linalg.inv(orthogonal).T
    return values, cartesian_to_fractional


def sample_ccp4(values, cartesian_to_fractional, xyz):
    """Trilinearly sample periodic CCP4 density at Cartesian coordinates."""
    xyz = np.asarray(xyz, dtype=np.float64)
    fractional = xyz @ cartesian_to_fractional
    nu, nv, nw = values.shape
    indices = [
        (fractional[:, 0] * nu) % nu,
        (fractional[:, 1] * nv) % nv,
        (fractional[:, 2] * nw) % nw,
    ]
    return scipy.ndimage.map_coordinates(
        values, indices, order=1, mode="wrap", prefilter=False
    )


def density_box(values, cartesian_to_fractional, center, grid_size=40, voxel_a=0.40):
    """Axis-aligned, whole-map-z-scored density box centred on the apo pocket anchor."""
    axis = (np.arange(grid_size) - (grid_size - 1) / 2) * voxel_a
    gx, gy, gz = np.meshgrid(axis, axis, axis, indexing="ij")
    xyz = np.column_stack([gx.ravel(), gy.ravel(), gz.ravel()]) + center
    sampled = sample_ccp4(values, cartesian_to_fractional, xyz)
    sampled_z = (sampled - values.mean()) / values.std()
    return xyz, sampled_z


def atom_coordinate_table(pair, values, cartesian_to_fractional):
    pocket = pair["apo"]["pocket"]
    xyz = as_numpy(pocket["coords"]).astype(float)
    channels = as_numpy(pocket["atoms_channel"]).astype(int)
    density = sample_ccp4(values, cartesian_to_fractional, xyz)
    density_z = (density - values.mean()) / values.std()
    return pd.DataFrame({
        "atom_index": np.arange(len(xyz)),
        "element": POCKET_ELEMENTS[channels],
        "channel": channels,
        "x_A": xyz[:, 0],
        "y_A": xyz[:, 1],
        "z_A": xyz[:, 2],
        "density": density,
        "density_z": density_z,
    })


def visualize_pair(pair_number=0, grid_size=40, voxel_a=0.40, density_sigma=1.0,
                   show_anchor=True):
    pair = pairs[pair_number]
    metadata = pair["metadata"]
    pocket = pair["apo"]["pocket"]
    anchor = pair["apo"]["anchor_ligand"]
    pocket_xyz = as_numpy(pocket["coords"]).astype(float)
    pocket_channels = as_numpy(pocket["atoms_channel"]).astype(int)
    anchor_xyz = as_numpy(anchor["coords"]).astype(float)
    center = as_numpy(anchor["center_coords"]).astype(float)

    map_path = Path(metadata["apo_map_path"])
    values, cartesian_to_fractional = load_ccp4_grid(map_path)
    box_xyz, box_z = density_box(
        values, cartesian_to_fractional, center, grid_size=grid_size, voxel_a=voxel_a
    )
    atom_table = atom_coordinate_table(pair, values, cartesian_to_fractional)

    upper = max(density_sigma + 0.5, float(np.nanpercentile(box_z, 99.7)))
    upper = min(upper, density_sigma + 5.0)
    fig = go.Figure()
    fig.add_trace(go.Isosurface(
        x=box_xyz[:, 0], y=box_xyz[:, 1], z=box_xyz[:, 2], value=box_z,
        isomin=density_sigma, isomax=upper, surface_count=3,
        colorscale=[[0, "#d9efff"], [1, "#2563eb"]],
        opacity=0.22, caps=dict(x_show=False, y_show=False, z_show=False),
        colorbar=dict(title="density z"), name="apo 2Fo–Fc density",
        hovertemplate="x=%{x:.2f} Å<br>y=%{y:.2f} Å<br>z=%{z:.2f} Å<br>z=%{value:.2f}<extra></extra>",
    ))

    for channel, element in enumerate(POCKET_ELEMENTS):
        keep = pocket_channels == channel
        if not keep.any():
            continue
        xyz = pocket_xyz[keep]
        density_z = atom_table.loc[keep, "density_z"].to_numpy()
        fig.add_trace(go.Scatter3d(
            x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
            marker=dict(size=ELEMENT_SIZES[element], color=ELEMENT_COLORS[element],
                        line=dict(width=0.3, color="#222")),
            customdata=density_z[:, None], name=f"apo pocket {element}",
            hovertemplate=(
                f"{element}<br>x=%{{x:.3f}} Å<br>y=%{{y:.3f}} Å<br>z=%{{z:.3f}} Å"
                "<br>density z=%{customdata[0]:.2f}<extra></extra>"
            ),
        ))

    if show_anchor:
        fig.add_trace(go.Scatter3d(
            x=anchor_xyz[:, 0], y=anchor_xyz[:, 1], z=anchor_xyz[:, 2], mode="markers",
            marker=dict(size=4.5, color="#dc2626", symbol="cross"),
            name="holo ligand anchor (not present)",
            hovertemplate="anchor<br>x=%{x:.3f} Å<br>y=%{y:.3f} Å<br>z=%{z:.3f} Å<extra></extra>",
        ))

    title = (
        f"{pair_number}: {metadata['reference_system_id']} → apo {metadata['apo_id']} | "
        f"fit RMSD {metadata['alignment']['rmsd']:.2f} Å | "
        f"pocket density {metadata['pocket_density_mean_z']:.2f} z"
    )
    fig.update_layout(
        title=title, template="plotly_white", height=760,
        legend=dict(x=0.01, y=0.99), margin=dict(l=0, r=0, b=0, t=55),
        scene=dict(
            aspectmode="data",
            xaxis_title="x (Å)", yaxis_title="y (Å)", zaxis_title="z (Å)",
            camera=dict(eye=dict(x=1.45, y=1.45, z=1.15)),
        ),
    )
    return pair, atom_table, fig

In [4]:
pair, atom_coordinates, figure = visualize_pair(
    pair_number=PAIR_NUMBER,
    grid_size=GRID_SIZE,
    voxel_a=VOXEL_A,
    density_sigma=DENSITY_SIGMA,
    show_anchor=SHOW_ANCHOR,
)

metadata = pair["metadata"]
display(pd.Series({
    "pair_id": pair["id"],
    "holo_system": metadata["reference_system_id"],
    "apo_chain": metadata["apo_id"],
    "apo_density_map": metadata["apo_map_path"],
    "apo_resolution_A": metadata["apo_resolution"],
    "alignment_rmsd_A": metadata["alignment"]["rmsd"],
    "pocket_atoms": len(atom_coordinates),
    "pocket_density_mean_z": metadata["pocket_density_mean_z"],
    "nearby_organic_atoms": metadata["nearby_organic_atoms"],
}, name="value").to_frame())

display(atom_coordinates.round(4))
figure.show()

,value
pair_id,3hfz__1__1.B__1.D__apo_1b7y_B
holo_system,3hfz__1__1.B__1.D
apo_chain,1b7y_B
apo_density_map,/home/shpark/prj-denovo/VoxBind/voxbind/datase...
apo_resolution_A,2.5
alignment_rmsd_A,0.344249
pocket_atoms,338
pocket_density_mean_z,2.851266
nearby_organic_atoms,0


,atom_index,element,channel,x_A,y_A,z_A,density,density_z
0,0,C,0,14.367,84.729,47.131,0.0665,0.5044
1,1,C,0,12.563,89.377,46.795,0.4659,3.5350
2,2,C,0,11.932,90.540,47.233,0.2170,1.6464
3,3,C,0,11.810,88.214,46.660,0.3050,2.3142
4,4,C,0,10.563,90.548,47.534,0.2018,1.5314
...,...,...,...,...,...,...,...,...
333,333,C,0,9.907,85.193,39.115,0.3228,2.4496
334,334,C,0,13.991,85.207,35.699,0.3797,2.8812
335,335,C,0,14.332,86.645,36.198,0.2440,1.8512
336,336,C,0,12.521,85.086,35.327,0.2397,1.8185


## Solvent-masked “dry pocket” approximation

An experimental apo map includes ordered waters. This panel makes the pocket visually as dry as possible by finding modeled apo water oxygens near the transferred ligand site and smoothly suppressing density within `DRY_MASK_OUTER_A` of those oxygens. Open cyan markers show every removed water site.

**This is a visualization-only approximation, not a separate experimental map.** The source CCP4 file and training data are never modified. Unmodeled/disordered solvent and model-phase effects cannot be uniquely subtracted from a 2Fo–Fc map.

In [5]:
DRY_SITE_RADIUS_A = 6.0   # include modeled waters this close to any anchor atom
DRY_MASK_INNER_A = 1.2   # completely suppress density inside this water-O radius
DRY_MASK_OUTER_A = 2.0   # smooth transition back to the untouched map


def resolve_apo_cif(pair):
    pdb_id = pair["metadata"]["apo_pdb_id"].lower()
    path = ROOT / "voxbind" / "dataset" / "data" / "cif" / f"{pdb_id}.cif"
    if not path.exists():
        raise FileNotFoundError(f"Missing deposited apo structure: {path}")
    return path


def modeled_apo_waters(pair, site_radius_a=6.0):
    """Water oxygen sites close to the transferred ligand in the apo structure."""
    anchor = as_numpy(pair["apo"]["anchor_ligand"]["coords"]).astype(float)
    structure = gemmi.read_structure(str(resolve_apo_cif(pair)))
    rows = []
    for chain in structure[0]:
        for residue in chain:
            if not residue.is_water():
                continue
            for atom in residue:
                if atom.element.name in ("H", "D"):
                    continue
                xyz = np.array([atom.pos.x, atom.pos.y, atom.pos.z], dtype=float)
                distance = float(np.linalg.norm(anchor - xyz, axis=1).min())
                if distance <= site_radius_a:
                    rows.append({
                        "chain": chain.name,
                        "residue": residue.name,
                        "seqid": str(residue.seqid),
                        "atom": atom.name.strip(),
                        "x_A": xyz[0], "y_A": xyz[1], "z_A": xyz[2],
                        "nearest_anchor_A": distance,
                    })
    columns = [
        "chain", "residue", "seqid", "atom", "x_A", "y_A", "z_A",
        "nearest_anchor_A",
    ]
    return pd.DataFrame(rows, columns=columns).sort_values(
        "nearest_anchor_A", ignore_index=True
    ) if rows else pd.DataFrame(columns=columns)


def smooth_water_exclusion_weight(xyz, water_xyz, inner_a=1.2, outer_a=2.0):
    """Return 0 at water sites, smoothly rising to 1 outside outer_a."""
    if len(water_xyz) == 0:
        return np.ones(len(xyz), dtype=np.float32)
    nearest = np.sqrt(
        ((xyz[:, None, :] - water_xyz[None, :, :]) ** 2).sum(axis=2).min(axis=1)
    )
    u = np.clip((nearest - inner_a) / max(outer_a - inner_a, 1e-6), 0.0, 1.0)
    return (u * u * (3.0 - 2.0 * u)).astype(np.float32)  # smoothstep


def visualize_dry_apo_pair(
    pair_number=0, grid_size=40, voxel_a=0.40, density_sigma=1.0,
    site_radius_a=6.0, mask_inner_a=1.2, mask_outer_a=2.0,
):
    pair = pairs[pair_number]
    metadata = pair["metadata"]
    pocket = pair["apo"]["pocket"]
    anchor = pair["apo"]["anchor_ligand"]
    pocket_xyz = as_numpy(pocket["coords"]).astype(float)
    pocket_channels = as_numpy(pocket["atoms_channel"]).astype(int)
    anchor_xyz = as_numpy(anchor["coords"]).astype(float)
    center = as_numpy(anchor["center_coords"]).astype(float)

    values, cartesian_to_fractional = load_ccp4_grid(Path(metadata["apo_map_path"]))
    box_xyz, original_z = density_box(
        values, cartesian_to_fractional, center,
        grid_size=grid_size, voxel_a=voxel_a,
    )
    waters = modeled_apo_waters(pair, site_radius_a=site_radius_a)
    water_xyz = waters[["x_A", "y_A", "z_A"]].to_numpy(dtype=float)
    weight = smooth_water_exclusion_weight(
        box_xyz, water_xyz, inner_a=mask_inner_a, outer_a=mask_outer_a,
    )
    dry_z = original_z * weight

    if len(waters):
        water_density = sample_ccp4(values, cartesian_to_fractional, water_xyz)
        waters["density_z"] = (water_density - values.mean()) / values.std()
    else:
        waters["density_z"] = pd.Series(dtype=float)

    upper = max(density_sigma + 0.5, float(np.nanpercentile(dry_z, 99.7)))
    upper = min(upper, density_sigma + 5.0)
    fig = go.Figure()
    fig.add_trace(go.Isosurface(
        x=box_xyz[:, 0], y=box_xyz[:, 1], z=box_xyz[:, 2], value=dry_z,
        isomin=density_sigma, isomax=upper, surface_count=3,
        colorscale=[[0, "#ede9fe"], [1, "#7c3aed"]], opacity=0.24,
        caps=dict(x_show=False, y_show=False, z_show=False),
        colorbar=dict(title="dry approx. z"), name="solvent-masked apo density",
        hovertemplate=("x=%{x:.2f} Å<br>y=%{y:.2f} Å<br>z=%{z:.2f} Å"
                       "<br>masked z=%{value:.2f}<extra></extra>"),
    ))

    for channel, element in enumerate(POCKET_ELEMENTS):
        keep = pocket_channels == channel
        if not keep.any():
            continue
        xyz = pocket_xyz[keep]
        fig.add_trace(go.Scatter3d(
            x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
            marker=dict(size=ELEMENT_SIZES[element], color=ELEMENT_COLORS[element],
                        line=dict(width=0.3, color="#222")),
            name=f"apo pocket {element}",
            hovertemplate=(f"{element}<br>x=%{{x:.3f}} Å<br>y=%{{y:.3f}} Å"
                           "<br>z=%{z:.3f} Å<extra></extra>"),
        ))

    if len(waters):
        custom = waters[["nearest_anchor_A", "density_z"]].to_numpy()
        fig.add_trace(go.Scatter3d(
            x=waters.x_A, y=waters.y_A, z=waters.z_A, mode="markers+text",
            marker=dict(size=7.5, color="#06b6d4", symbol="circle-open",
                        line=dict(width=2.0, color="#0891b2")),
            text=[f"HOH {seqid}" for seqid in waters.seqid], textposition="top center",
            customdata=custom, name="modeled waters removed from density view",
            hovertemplate=("water O<br>x=%{x:.3f} Å<br>y=%{y:.3f} Å<br>z=%{z:.3f} Å"
                           "<br>anchor distance=%{customdata[0]:.2f} Å"
                           "<br>original density z=%{customdata[1]:.2f}<extra></extra>"),
        ))

    fig.add_trace(go.Scatter3d(
        x=anchor_xyz[:, 0], y=anchor_xyz[:, 1], z=anchor_xyz[:, 2], mode="markers",
        marker=dict(size=4.5, color="#dc2626", symbol="cross"),
        name="holo ligand anchor (not present)",
        hovertemplate="anchor<br>x=%{x:.3f} Å<br>y=%{y:.3f} Å<br>z=%{z:.3f} Å<extra></extra>",
    ))

    changed = weight < 0.999
    diagnostics = {
        "modeled_waters_removed": len(waters),
        "box_voxels_affected": int(changed.sum()),
        "box_fraction_affected": float(changed.mean()),
        "mask_inner_A": mask_inner_a,
        "mask_outer_A": mask_outer_a,
    }
    fig.update_layout(
        title=(f"{pair_number}: apo {metadata['apo_id']} — solvent-masked dry approximation "
               f"({len(waters)} modeled waters)"),
        template="plotly_white", height=760,
        legend=dict(x=0.01, y=0.99), margin=dict(l=0, r=0, b=0, t=55),
        scene=dict(
            aspectmode="data", xaxis_title="x (Å)", yaxis_title="y (Å)",
            zaxis_title="z (Å)", camera=dict(eye=dict(x=1.45, y=1.45, z=1.15)),
        ),
    )
    return pair, waters, fig, diagnostics

In [6]:
dry_pair, removed_waters, dry_figure, dry_diagnostics = visualize_dry_apo_pair(
    pair_number=PAIR_NUMBER,
    grid_size=GRID_SIZE,
    voxel_a=VOXEL_A,
    density_sigma=DENSITY_SIGMA,
    site_radius_a=DRY_SITE_RADIUS_A,
    mask_inner_a=DRY_MASK_INNER_A,
    mask_outer_a=DRY_MASK_OUTER_A,
)

display(pd.Series({
    "pair_id": dry_pair["id"],
    "source_apo_map": dry_pair["metadata"]["apo_map_path"],
    "interpretation": "visualization-only solvent-masked approximation",
    **dry_diagnostics,
}, name="value").to_frame())
display(removed_waters.round(4))
dry_figure.show()

,value
pair_id,3hfz__1__1.B__1.D__apo_1b7y_B
source_apo_map,/home/shpark/prj-denovo/VoxBind/voxbind/datase...
interpretation,visualization-only solvent-masked approximation
modeled_waters_removed,5
box_voxels_affected,2484
box_fraction_affected,0.038812
mask_inner_A,1.2
mask_outer_A,2.0


,chain,residue,seqid,atom,x_A,y_A,z_A,nearest_anchor_A,density_z
0,B,HOH,788,O,4.931,85.201,42.282,0.7260,3.2302
1,B,HOH,892,O,-0.203,84.939,40.158,1.1933,2.5420
2,B,HOH,858,O,0.987,82.761,37.090,3.7464,1.0743
3,B,HOH,807,O,2.744,88.111,36.692,3.9047,0.8571
4,B,HOH,900,O,0.478,84.957,35.936,4.0763,1.1872


## Matching holo pocket: ligand + experimental density

The holo structure is shown in its original deposited coordinate frame with its real ligand and its own experimental 2Fo–Fc map. The paired records store holo coordinates aligned into the apo frame; the helper below applies the exact inverse Kabsch transform before sampling the holo map. Thus neither panel mixes coordinates from one state with density from the other.

In [7]:
LIGAND_ELEMENTS = np.array(["C", "O", "N", "S", "F", "Cl", "P", "Other"])
LIGAND_COLORS = {
    "C": "#303030", "O": "#e74c3c", "N": "#2563eb", "S": "#eab308",
    "F": "#22c55e", "Cl": "#16a34a", "P": "#f97316", "Other": "#a855f7",
}


def resolve_holo_map(pair):
    """Find the CCP4 map belonging to the holo PDB in this pair."""
    metadata = pair["metadata"]
    recorded = metadata.get("holo_map_path")
    if recorded and Path(recorded).exists():
        return Path(recorded)
    pdb_id = metadata["holo_pdb_id"].lower()
    candidates = [
        ROOT / "voxbind" / "dataset" / "data" / "ccp4" / f"{pdb_id}.ccp4",
        ROOT / "voxbind" / "dataset" / "data" / "plinder" / "ccp4" / f"{pdb_id}.ccp4",
    ]
    found = next((path for path in candidates if path.exists()), None)
    if found is None:
        raise FileNotFoundError(f"No experimental holo map found for {pdb_id}: {candidates}")
    return found


def aligned_apo_to_deposited_holo(xyz, metadata):
    """Invert x_apo = x_holo @ R.T + t from the paired-record alignment."""
    xyz = as_numpy(xyz).astype(np.float64)
    rotation = as_numpy(metadata["alignment"]["R"]).astype(np.float64)
    translation = as_numpy(metadata["alignment"]["t"]).astype(np.float64)
    return (xyz - translation) @ rotation


def structure_coordinate_table(role, xyz, channels, elements, values,
                               cartesian_to_fractional):
    xyz = np.asarray(xyz, dtype=float)
    channels = np.asarray(channels, dtype=int)
    density = sample_ccp4(values, cartesian_to_fractional, xyz)
    density_z = (density - values.mean()) / values.std()
    return pd.DataFrame({
        "role": role,
        "atom_index": np.arange(len(xyz)),
        "element": elements[channels],
        "channel": channels,
        "x_A": xyz[:, 0], "y_A": xyz[:, 1], "z_A": xyz[:, 2],
        "density": density, "density_z": density_z,
    })


def visualize_holo_pair(pair_number=0, grid_size=40, voxel_a=0.40,
                        density_sigma=1.0):
    pair = pairs[pair_number]
    metadata = pair["metadata"]
    pocket = pair["holo"]["pocket"]
    ligand = pair["holo"]["ligand"]

    # Stored holo coordinates were transformed into the apo frame for paired
    # comparisons. Return them to the deposited holo/map frame here.
    pocket_xyz = aligned_apo_to_deposited_holo(pocket["coords"], metadata)
    ligand_xyz = aligned_apo_to_deposited_holo(ligand["coords"], metadata)
    pocket_channels = as_numpy(pocket["atoms_channel"]).astype(int)
    ligand_channels = as_numpy(ligand["atoms_channel"]).astype(int)
    center = ligand_xyz.mean(axis=0)

    map_path = resolve_holo_map(pair)
    values, cartesian_to_fractional = load_ccp4_grid(map_path)
    box_xyz, box_z = density_box(
        values, cartesian_to_fractional, center,
        grid_size=grid_size, voxel_a=voxel_a,
    )
    pocket_table = structure_coordinate_table(
        "holo pocket", pocket_xyz, pocket_channels, POCKET_ELEMENTS,
        values, cartesian_to_fractional,
    )
    ligand_table = structure_coordinate_table(
        "holo ligand", ligand_xyz, ligand_channels, LIGAND_ELEMENTS,
        values, cartesian_to_fractional,
    )
    coordinate_table = pd.concat([pocket_table, ligand_table], ignore_index=True)

    upper = max(density_sigma + 0.5, float(np.nanpercentile(box_z, 99.7)))
    upper = min(upper, density_sigma + 5.0)
    fig = go.Figure()
    fig.add_trace(go.Isosurface(
        x=box_xyz[:, 0], y=box_xyz[:, 1], z=box_xyz[:, 2], value=box_z,
        isomin=density_sigma, isomax=upper, surface_count=3,
        colorscale=[[0, "#dcfce7"], [1, "#15803d"]], opacity=0.22,
        caps=dict(x_show=False, y_show=False, z_show=False),
        colorbar=dict(title="holo density z"), name="holo 2Fo–Fc density",
        hovertemplate=("x=%{x:.2f} Å<br>y=%{y:.2f} Å<br>z=%{z:.2f} Å"
                       "<br>z=%{value:.2f}<extra></extra>"),
    ))

    for channel, element in enumerate(POCKET_ELEMENTS):
        keep = pocket_channels == channel
        if not keep.any():
            continue
        xyz = pocket_xyz[keep]
        density_z = pocket_table.loc[keep, "density_z"].to_numpy()
        fig.add_trace(go.Scatter3d(
            x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
            marker=dict(size=3.8, color=ELEMENT_COLORS[element], opacity=0.72,
                        line=dict(width=0.2, color="#222")),
            customdata=density_z[:, None], name=f"holo pocket {element}",
            hovertemplate=(f"pocket {element}<br>x=%{{x:.3f}} Å<br>y=%{{y:.3f}} Å"
                           "<br>z=%{z:.3f} Å<br>density z=%{customdata[0]:.2f}"
                           "<extra></extra>"),
        ))

    for channel, element in enumerate(LIGAND_ELEMENTS):
        keep = ligand_channels == channel
        if not keep.any():
            continue
        xyz = ligand_xyz[keep]
        density_z = ligand_table.loc[keep, "density_z"].to_numpy()
        fig.add_trace(go.Scatter3d(
            x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
            marker=dict(size=6.5, color=LIGAND_COLORS[element], symbol="diamond",
                        line=dict(width=0.7, color="#111")),
            customdata=density_z[:, None], name=f"real holo ligand {element}",
            hovertemplate=(f"ligand {element}<br>x=%{{x:.3f}} Å<br>y=%{{y:.3f}} Å"
                           "<br>z=%{z:.3f} Å<br>density z=%{customdata[0]:.2f}"
                           "<extra></extra>"),
        ))

    fig.update_layout(
        title=(f"{pair_number}: holo {metadata['holo_pdb_id']} — real ligand + "
               "matching experimental 2Fo–Fc density"),
        template="plotly_white", height=760,
        legend=dict(x=0.01, y=0.99), margin=dict(l=0, r=0, b=0, t=55),
        scene=dict(
            aspectmode="data", xaxis_title="x (Å)", yaxis_title="y (Å)",
            zaxis_title="z (Å)", camera=dict(eye=dict(x=1.45, y=1.45, z=1.15)),
        ),
    )
    return pair, coordinate_table, fig, map_path

In [8]:
holo_pair, holo_coordinates, holo_figure, holo_map = visualize_holo_pair(
    pair_number=PAIR_NUMBER,
    grid_size=GRID_SIZE,
    voxel_a=VOXEL_A,
    density_sigma=DENSITY_SIGMA,
)

holo_metadata = holo_pair["metadata"]
display(pd.Series({
    "pair_id": holo_pair["id"],
    "holo_pdb": holo_metadata["holo_pdb_id"],
    "holo_density_map": str(holo_map),
    "coordinate_frame": "original deposited holo/map frame",
    "holo_pocket_atoms": int((holo_coordinates.role == "holo pocket").sum()),
    "real_holo_ligand_atoms": int((holo_coordinates.role == "holo ligand").sum()),
    "ligand_density_mean_z": float(
        holo_coordinates.loc[holo_coordinates.role == "holo ligand", "density_z"].mean()
    ),
}, name="value").to_frame())

display(holo_coordinates.round(4))
holo_figure.show()

,value
pair_id,3hfz__1__1.B__1.D__apo_1b7y_B
holo_pdb,3hfz
holo_density_map,/home/shpark/prj-denovo/VoxBind/voxbind/datase...
coordinate_frame,original deposited holo/map frame
holo_pocket_atoms,339
real_holo_ligand_atoms,13
ligand_density_mean_z,1.626193


,role,atom_index,element,channel,x_A,y_A,z_A,density,density_z
0,holo pocket,0,O,1,14.376,80.774,45.789,0.1040,0.9216
1,holo pocket,1,C,0,12.778,88.482,46.984,0.2818,2.4960
2,holo pocket,2,C,0,12.089,89.620,47.376,0.1315,1.1653
3,holo pocket,3,C,0,12.078,87.296,46.848,0.2337,2.0705
4,holo pocket,4,C,0,10.723,89.576,47.623,0.2532,2.2429
...,...,...,...,...,...,...,...,...,...
347,holo ligand,8,C,0,5.933,84.511,42.172,0.2834,2.5110
348,holo ligand,9,O,1,4.484,83.921,43.964,0.1308,1.1591
349,holo ligand,10,C,0,0.108,85.311,39.804,0.2335,2.0684
350,holo ligand,11,O,1,-1.003,84.753,39.900,0.2033,1.8013


## Find a specific structure

Search the compact index, copy its row number into `PAIR_NUMBER`, and rerun the visualization cell.

In [9]:
QUERY = ""  # examples: "1d2a", "1d1q", or part of a PLINDER system ID

searchable = pair_index.astype(str).agg(" ".join, axis=1)
matches = pair_index[searchable.str.contains(QUERY, case=False, regex=False)] if QUERY else pair_index
display(matches.head(50))

,alignment_coverage,alignment_identity,apo_chain,apo_id,apo_pdb_id,apo_resolution,binding_site_rmsd,holo_chain,n_alignment_atoms,n_apo_pocket_atoms,nearby_hetero_atoms,nearby_organic_atoms,pair_id,pocket_density_coverage,pocket_density_mean_z,pocket_fident,pocket_lddt,reason,reference_system_id,status
0,1.000000,1.000000,B,1b7y_B,1b7y,2.50,0.344249,B,396,338,0,0,3hfz__1__1.B__1.D__apo_1b7y_B,0.973373,2.851266,100.0,100.0,ok,3hfz__1__1.B__1.D,ok
1,1.000000,1.000000,B,1b7y_B,1b7y,2.50,0.344249,B,396,338,0,0,3hfz__1__2.B__2.D__apo_1b7y_B,0.973373,2.851266,100.0,100.0,ok,3hfz__1__2.B__2.D,ok
2,1.000000,1.000000,B,1d09_B,1d09,2.10,1.037715,B,196,283,0,0,4fyv__1__1.B__1.G__apo_1d09_B,0.830389,1.232072,100.0,88.0,ok,4fyv__1__1.B__1.G,ok
3,1.000000,1.000000,B,1d09_B,1d09,2.10,1.175960,D,208,288,0,0,4fyv__1__1.D__1.I__apo_1d09_B,0.843750,1.260750,100.0,89.0,ok,4fyv__1__1.D__1.I,ok
4,1.000000,1.000000,B,1d09_B,1d09,2.10,1.037715,B,196,283,0,0,4fyv__1__2.B__2.G__apo_1d09_B,0.830389,1.232072,100.0,88.0,ok,4fyv__1__2.B__2.G,ok
5,1.000000,1.000000,B,1d09_B,1d09,2.10,1.175960,D,208,288,0,0,4fyv__1__2.D__2.I__apo_1d09_B,0.843750,1.260750,100.0,89.0,ok,4fyv__1__2.D__2.I,ok
6,1.000000,1.000000,B,1d09_B,1d09,2.10,1.037715,B,196,283,0,0,4fyv__1__3.B__3.G__apo_1d09_B,0.830389,1.232072,100.0,88.0,ok,4fyv__1__3.B__3.G,ok
7,1.000000,1.000000,B,1d09_B,1d09,2.10,1.175960,D,208,288,0,0,4fyv__1__3.D__3.I__apo_1d09_B,0.843750,1.260750,100.0,89.0,ok,4fyv__1__3.D__3.I,ok
8,1.000000,1.000000,A,1d2a_A,1d2a,1.90,0.370830,A,280,284,5,0,1d1q__1__1.A__1.C__apo_1d2a_A,1.000000,2.583288,100.0,98.0,ok,1d1q__1__1.A__1.C,ok
9,1.000000,1.000000,B,1fm8_B,1fm8,2.30,0.575094,A,352,366,0,0,1eyq__1__1.A__1.F__apo_1fm8_B,0.937158,2.159976,100.0,97.0,ok,1eyq__1__1.A__1.F,ok
